# All-event PCS evolution and temporal-alignment null test

This notebook is converted from `012_all_event_and_test.py`. It combines:

- cumulative occurrence curves of the 35 PCS-selected states;
- hourly occurrence heatmaps;
- the circular-shift null distribution of terminal enrichment; and
- observed event-aligned occupancy with the pointwise null envelope.

Run the cells from top to bottom. The notebook reads existing PCS, DEC, and null-test outputs; it does not recompute PCS or the circular-shift test.

## 1. Imports

In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## 2. Paths and publication settings

Adjust `PROJECT_ROOT`, `NULL_DIR`, or `OUTPUT_DIR` here if the notebook is moved.

In [ ]:
# Path configuration
# Keep the notebook in the original `uilt_paper_figure` directory for these
# defaults to match the Python script. If it is stored elsewhere, set
# PROJECT_ROOT explicitly to the Illgraben project root.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]

DEC_RESULT_DIR = PROJECT_ROOT / "uilt_oneday_all" / "02clustering" / "result" / "DEC_result"
PARAMETER_DIR = DEC_RESULT_DIR / "all_parement"
PCS_WORKBOOK = DEC_RESULT_DIR / "precursor_curve_metrics" / "precursor_clusters_all_metrics.xlsx"
DEFAULT_NULL_DIR = NOTEBOOK_DIR / "Fig_PCS_time_shift_null"
DEFAULT_OUTPUT_DIR = NOTEBOOK_DIR / "Fig_all_event_and_null_test"

# Edit these two paths when needed.
NULL_DIR = DEFAULT_NULL_DIR
OUTPUT_DIR = DEFAULT_OUTPUT_DIR

EXPECTED_EVENTS = 44
EXPECTED_SELECTED = 35
N_HOURS = 24

BLUE = "#0F4D92"
BLUE_LIGHT = "#8CB9D9"
ORANGE = "#D97732"
RED = "#B64342"
GREY = "#CFCECE"
DARK_GREY = "#4D4D4D"

PUBLICATION_STYLE = {
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8.5,
    "axes.titlesize": 9.5,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.2,
    "legend.fontsize": 7.0,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.9,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}


## 3. Data-loading and utility functions

In [ ]:
def event_number(event_name: str) -> int:
    match = re.search(r"(\d+)$", str(event_name))
    return int(match.group(1)) if match else 10**9

def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path}")

def load_selected_states() -> pd.DataFrame:
    """Load the authoritative 1 h-terminal-window PCS cohort."""
    require_file(PCS_WORKBOOK)
    selected = pd.read_excel(PCS_WORKBOOK, sheet_name="pcs")
    required = {"event", "cluster", "selection_status"}
    missing = required.difference(selected.columns)
    if missing:
        raise KeyError(f"Missing columns in PCS workbook: {sorted(missing)}")

    selected["event"] = selected["event"].astype(str)
    selected["cluster"] = pd.to_numeric(selected["cluster"], errors="coerce").fillna(-1).astype(int)
    selected = selected.loc[
        selected["selection_status"].eq("selected") & selected["cluster"].ge(0)
    ].copy()
    selected["event_number"] = selected["event"].map(event_number)
    selected = selected.sort_values("event_number").reset_index(drop=True)

    total_rows = pd.read_excel(PCS_WORKBOOK, sheet_name="pcs", usecols=["event"])
    if total_rows["event"].nunique() != EXPECTED_EVENTS:
        raise ValueError(
            f"Expected {EXPECTED_EVENTS} events in the PCS table, "
            f"found {total_rows['event'].nunique()}."
        )
    if len(selected) != EXPECTED_SELECTED:
        raise ValueError(
            f"Expected {EXPECTED_SELECTED} PCS-selected events, found {len(selected)}."
        )
    return selected

def load_event_state(event_name: str, cluster_id: int) -> Tuple[np.ndarray, np.ndarray]:
    path = PARAMETER_DIR / f"{event_name}_all_parement.npz"
    require_file(path)
    with np.load(path, allow_pickle=True) as archive:
        labels = np.asarray(archive["y_pred_last"], dtype=int)
    relative_hours = np.arange(len(labels), dtype=float) / 60.0 - 24.0
    return relative_hours, labels == int(cluster_id)

def build_all_event_data(
    selected: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, List[str], np.ndarray]:
    """Return minute cumulative curves and hourly counts for all selected states."""
    cumulative_rows: List[np.ndarray] = []
    hourly_rows: List[np.ndarray] = []
    event_labels: List[str] = []
    minute_axis = np.arange(1441, dtype=float) / 60.0 - 24.0
    hour_edges = np.arange(-24, 1, 1, dtype=float)

    for row in selected.itertuples(index=False):
        relative_hours, state_mask = load_event_state(row.event, int(row.cluster))
        valid = (relative_hours >= -24.0) & (relative_hours < 0.0)
        relative_hours = relative_hours[valid]
        state_mask = state_mask[valid]

        minute_index = np.floor((relative_hours + 24.0) * 60.0).astype(int)
        minute_index = np.clip(minute_index, 0, 1439)
        minute_counts = np.bincount(minute_index[state_mask], minlength=1440)
        cumulative_rows.append(np.r_[0.0, np.cumsum(minute_counts, dtype=float)])

        hourly_counts, _ = np.histogram(relative_hours[state_mask], bins=hour_edges)
        hourly_rows.append(hourly_counts.astype(float))
        event_labels.append(f"E{event_number(row.event)}")

    cumulative = np.stack(cumulative_rows)
    hourly = np.stack(hourly_rows)

    # Retain event-number order, matching the original all-event figure.
    return cumulative, hourly, event_labels, minute_axis

def load_null_outputs(null_dir: Path) -> Dict[str, np.ndarray]:
    global_path = null_dir / "pcs_time_shift_global_null.csv"
    hourly_path = null_dir / "pcs_time_shift_hourly_curve.csv"
    summary_path = null_dir / "pcs_time_shift_null_summary.csv"
    for path in (global_path, hourly_path, summary_path):
        require_file(path)

    global_null = pd.read_csv(global_path)
    hourly = pd.read_csv(hourly_path)
    summary = pd.read_csv(summary_path)
    enrichment_row = summary.loc[
        summary["test"].eq("Median terminal enrichment of PCS-selected states")
    ]
    if len(enrichment_row) != 1:
        raise ValueError("Could not identify the terminal-enrichment summary row.")
    enrichment_row = enrichment_row.iloc[0]

    return {
        "null_enrichment": global_null["median_terminal_enrichment"].to_numpy(float),
        "observed_enrichment": float(enrichment_row["observed"]),
        "null_enrichment_median": float(enrichment_row["null_median"]),
        "null_enrichment_low": float(enrichment_row["null_2.5%"]),
        "null_enrichment_high": float(enrichment_row["null_97.5%"]),
        "p_enrichment": float(enrichment_row["one_sided_monte_carlo_p"]),
        "time_hours": hourly["time_before_initiation_h"].to_numpy(float),
        "observed_occupancy": hourly["observed_mean_occupancy"].to_numpy(float),
        "null_occupancy_median": hourly["null_mean_occupancy_median"].to_numpy(float),
        "null_occupancy_low": hourly["null_mean_occupancy_2.5%"].to_numpy(float),
        "null_occupancy_high": hourly["null_mean_occupancy_97.5%"].to_numpy(float),
    }

def format_pvalue(value: float) -> str:
    return "<0.0001" if value < 0.0001 else f"={value:.4f}"

def add_panel_label(ax: plt.Axes, label: str, x: float = -0.09, y: float = 1.08) -> None:
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="top",
        ha="left",
    )


## 5. Load and prepare all-event data

In [ ]:
selected = load_selected_states()
cumulative, hourly, heatmap_labels, minute_axis = build_all_event_data(selected)
null_data = load_null_outputs(NULL_DIR.resolve())

print(f"PCS-selected events: {len(selected)}/{EXPECTED_EVENTS}")
print(f"Cumulative array shape: {cumulative.shape}")
print(f"Hourly heatmap shape: {hourly.shape}")


## 6. Generate and export the combined figure

Outputs are saved as PNG, PDF, SVG, and TIFF, with CSV source data for panels a and b.

In [ ]:
# ---------- Editable: colours, panel-label positions, typography, dimensions, spacing ----------
MAIN_COLORS = {
    "blue":        "#0F4D92",
    "blue_light":  "#8CB9D9",
    "orange":      "#D97732",
    "red":         "#B64342",
    "grey":        "#CFCECE",
    "dark_grey":   "#4D4D4D",
    "black":       "#272727",
    "cumulative_colormap": "tab20",
    "heatmap_colormap":    "YlOrRd",
    "grid_line":           "0.88",
    "grid_alpha":          0.8,
}
MAIN_CUMULATIVE = dict(linewidth=0.8, alpha=0.90)
MAIN_HEATMAP_COLORBAR = dict(orientation="horizontal", pad=0.10, fraction=0.075, aspect=28, label_pad=2)
MAIN_ENRICHMENT_HIST = dict(bins=35, density=True, edgecolor="white", linewidth=0.45, ci_alpha=0.07)
MAIN_OBSERVED_LINE = dict(color_key="red", linewidth=2.0)
MAIN_OCCUPANCY_OBSERVED = dict(linewidth=1.9, marker="o", markersize=2.3)
MAIN_OCCUPANCY_NULL_MEDIAN = dict(linestyle="--", linewidth=1.2)
MAIN_OCCUPANCY_NULL_CI = dict(alpha=0.60)
MAIN_PANEL_LABEL_POS = {
    "a": dict(x=-0.14, y=1.08),
    "b": dict(x=-0.14, y=1.06),
    "c": dict(x=-0.16, y=1.08),
    "d": dict(x=-0.16, y=1.08),
}
MAIN_PANEL_LABEL_STYLE = dict(fontsize=12, fontweight="bold", va="top", ha="left")
MAIN_HEATMAP_YTICK_FONTSIZE = 4.5
MAIN_STATS_TEXT = dict(fontsize=7.1, va="top", ha="left", position=(0.03, 0.96))
MAIN_OCCUPANCY_LEGEND_ORDER = [2, 1, 0]
MAIN_OCCUPANCY_LEGEND_LOC = "upper left"
MAIN_FIGSIZE = (7.2, 6.0)
MAIN_FONT = {
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Microsoft YaHei", "SimHei", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.titlesize": 9.5,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 7.7,
    "legend.fontsize": 7.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.9,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}
MAIN_GRID = dict(height_ratios=[1.0, 1.35], width_ratios=[1.10, 1.0], hspace=0.22, wspace=0.22)
MAIN_AXIS = dict(
    x_lim=(-24, 0),
    x_ticks=list(range(-24, 1, 6)),
    x_label="Time before debris-flow initiation (h)",
    final_hour_span=(-1, 0),
    final_hour_alpha=0.10,
    initiation_linewidth=0.75,
)
MAIN_GRID_LINES = dict(color=MAIN_COLORS["grid_line"], linewidth=0.45, alpha=MAIN_COLORS["grid_alpha"])
MAIN_YLABELS = dict(
    panel_a="Cumulative count",
    panel_b="Event",
    panel_c="Probability density",
    panel_d="Mean state occupancy",
)
MAIN_OUT_FORMATS = ("svg", "pdf", "tiff", "png")
MAIN_OUT_DPI = 600
MAIN_OUT_STEM = "Figure_4"
# --------------------------------------------------------------------------------------------------------

def add_panel_label_local(ax, label, x=-0.14, y=1.08):
    ax.text(x, y, label, transform=ax.transAxes, **MAIN_PANEL_LABEL_STYLE)

def format_pvalue_local(value: float) -> str:
    return "<0.0001" if value < 0.0001 else f"={value:.4f}"


plt.rcParams.update(MAIN_FONT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fig = plt.figure(figsize=MAIN_FIGSIZE)
grid = fig.add_gridspec(2, 2, **MAIN_GRID)
ax_a = fig.add_subplot(grid[0, 0])
ax_b = fig.add_subplot(grid[1, 0])
ax_c = fig.add_subplot(grid[0, 1])
ax_d = fig.add_subplot(grid[1, 1])

# Panel a — All-event cumulative curves
n_curves = len(cumulative)
line_cmap = plt.get_cmap(MAIN_COLORS["cumulative_colormap"], n_curves)
for index, curve in enumerate(cumulative):
    ax_a.plot(minute_axis, curve, color=line_cmap(index), **MAIN_CUMULATIVE)
ax_a.set_xlim(*MAIN_AXIS["x_lim"])
ax_a.set_xticks(MAIN_AXIS["x_ticks"])
ax_a.set_xlabel(MAIN_AXIS["x_label"])
ax_a.set_ylabel(MAIN_YLABELS["panel_a"])
ax_a.grid(**MAIN_GRID_LINES)
add_panel_label_local(ax_a, "a", **MAIN_PANEL_LABEL_POS["a"])

# Panel b — Hourly all-event heatmap
n_rows = len(heatmap_labels)
image = ax_b.imshow(
    hourly,
    aspect="auto",
    interpolation="nearest",
    cmap=MAIN_COLORS["heatmap_colormap"],
    extent=[-24, 0, n_rows - 0.5, -0.5],
    vmin=0,
)
ax_b.set_xlim(*MAIN_AXIS["x_lim"])
ax_b.set_xticks(MAIN_AXIS["x_ticks"])
ax_b.set_xlabel(MAIN_AXIS["x_label"])
ax_b.set_ylabel(MAIN_YLABELS["panel_b"])
y_ticks = np.arange(n_rows)
ax_b.set_yticks(y_ticks)
ax_b.set_yticklabels([heatmap_labels[i] for i in y_ticks], fontsize=MAIN_HEATMAP_YTICK_FONTSIZE)
colorbar = fig.colorbar(image, ax=ax_b, **{k: v for k, v in MAIN_HEATMAP_COLORBAR.items() if k != "label_pad"})
colorbar.set_label("Hourly count", labelpad=MAIN_HEATMAP_COLORBAR["label_pad"])
add_panel_label_local(ax_b, "b", **MAIN_PANEL_LABEL_POS["b"])

# Panel c — Terminal-enrichment null distribution
enrichment = null_data["null_enrichment"]
enrichment = enrichment[np.isfinite(enrichment)]
ax_c.hist(
    enrichment,
    bins=MAIN_ENRICHMENT_HIST["bins"],
    density=MAIN_ENRICHMENT_HIST["density"],
    color=MAIN_COLORS["blue_light"],
    edgecolor=MAIN_ENRICHMENT_HIST["edgecolor"],
    linewidth=MAIN_ENRICHMENT_HIST["linewidth"],
)
ax_c.axvspan(
    null_data["null_enrichment_low"], null_data["null_enrichment_high"],
    color=MAIN_COLORS["blue"], alpha=MAIN_ENRICHMENT_HIST["ci_alpha"], linewidth=0,
)
ax_c.axvline(
    null_data["observed_enrichment"],
    color=MAIN_COLORS[MAIN_OBSERVED_LINE["color_key"]],
    linewidth=MAIN_OBSERVED_LINE["linewidth"],
)
ax_c.set_xlabel("Median final-hour enrichment")
ax_c.set_ylabel(MAIN_YLABELS["panel_c"])
ax_c.text(
    *MAIN_STATS_TEXT["position"],
    (
        f"Observed: {null_data['observed_enrichment']:.2f}x\n"
        f"Null median [95%]: {null_data['null_enrichment_median']:.2f} "
        f"[{null_data['null_enrichment_low']:.2f}, {null_data['null_enrichment_high']:.2f}]\n"
        f"Monte Carlo p {format_pvalue_local(null_data['p_enrichment'])}"
    ),
    transform=ax_c.transAxes,
    va=MAIN_STATS_TEXT["va"],
    ha=MAIN_STATS_TEXT["ha"],
    fontsize=MAIN_STATS_TEXT["fontsize"],
)
add_panel_label_local(ax_c, "c", **MAIN_PANEL_LABEL_POS["c"])

# Panel d — Event-aligned occupancy curve
hours = null_data["time_hours"]
ax_d.fill_between(
    hours,
    null_data["null_occupancy_low"],
    null_data["null_occupancy_high"],
    color=MAIN_COLORS["grey"],
    alpha=MAIN_OCCUPANCY_NULL_CI["alpha"],
    linewidth=0,
    label="Null 95% interval",
)
ax_d.plot(
    hours, null_data["null_occupancy_median"],
    color=MAIN_COLORS["dark_grey"],
    linestyle=MAIN_OCCUPANCY_NULL_MEDIAN["linestyle"],
    linewidth=MAIN_OCCUPANCY_NULL_MEDIAN["linewidth"],
    label="Null median",
)
ax_d.plot(
    hours, null_data["observed_occupancy"],
    color=MAIN_COLORS["blue"],
    linewidth=MAIN_OCCUPANCY_OBSERVED["linewidth"],
    marker=MAIN_OCCUPANCY_OBSERVED["marker"],
    markersize=MAIN_OCCUPANCY_OBSERVED["markersize"],
    label="Observed alignment",
)
ax_d.axvspan(*MAIN_AXIS["final_hour_span"], color=MAIN_COLORS["orange"], alpha=MAIN_AXIS["final_hour_alpha"], linewidth=0)
ax_d.axvline(0, color=MAIN_COLORS["black"], linewidth=MAIN_AXIS["initiation_linewidth"])
ax_d.set_xlim(*MAIN_AXIS["x_lim"])
ax_d.set_xticks(MAIN_AXIS["x_ticks"])
ax_d.set_xlabel(MAIN_AXIS["x_label"])
ax_d.set_ylabel(MAIN_YLABELS["panel_d"])
handles, labels = ax_d.get_legend_handles_labels()
order = MAIN_OCCUPANCY_LEGEND_ORDER
ax_d.legend([handles[i] for i in order], [labels[i] for i in order], loc=MAIN_OCCUPANCY_LEGEND_LOC)
add_panel_label_local(ax_d, "d", **MAIN_PANEL_LABEL_POS["d"])

output_stem = OUTPUT_DIR / MAIN_OUT_STEM
for fmt in MAIN_OUT_FORMATS:
    ext = "." + fmt
    save_kwargs = dict(dpi=MAIN_OUT_DPI, bbox_inches="tight", facecolor="white")
    if fmt == "tiff":
        save_kwargs["pil_kwargs"] = {"compression": "tiff_lzw"}
    fig.savefig(str(output_stem) + ext, **save_kwargs)

try:
    display(fig)
except Exception:
    pass
plt.close(fig)

print("Saved main figure a-d:")
for fmt in MAIN_OUT_FORMATS:
    print(f"  {str(output_stem)}.{fmt}")


In [ ]:
# ---------- Editable: colours, panel-label positions, typography, dimensions, spacing ----------
MAIN_COLORS = {
    "blue":        "#0F4D92",
    "blue_light":  "#8CB9D9",
    "orange":      "#D97732",
    "red":         "#B64342",
    "grey":        "#CFCECE",
    "dark_grey":   "#4D4D4D",
    "black":       "#272727",
    "cumulative_colormap": "tab20",
    "heatmap_colormap":    "YlOrRd",
    "grid_line":           "0.88",
    "grid_alpha":          0.8,
}
MAIN_CUMULATIVE = dict(linewidth=0.8, alpha=0.90)
# ==============================================================
# ★ heatmap colorbar —— 精确位置 + 垂直色条 + 字号/字体可配
# ==============================================================
MAIN_HEATMAP_COLORBAR = dict(
    use_absolute_position=True,
    absolute_rect=(0.92, 0.525, 0.015, 0.355),          # (left, bottom, width, height)
    orientation="vertical",
    label="Hourly count",
    tick_labelsize=8,                                 # colorbar 刻度字号
    ylabelsize=10,                                       # colorbar ylabel 字号
    labelpad=6,
    font_family=["Arial"],
    # ---- use_absolute_position=False（挤出版）备用 ----
    pad=0.10, fraction=0.075, aspect=28,
)
# ==============================================================
# ★ Panel b (heatmap) ytick 显示策略：每 N 个 event 显示 1 个，字号和其它 3 张图一致
# ==============================================================
MAIN_HEATMAP_YTICK_EVERY = 2                             # ★ 每 3 个 event 显示 1 个 label；改成 1=全显示，2=每 2 个，5=每 5 个
MAIN_HEATMAP_YTICK_FONTSIZE = 7.9                        # ★ 与 MAIN_FONT["ytick.labelsize"] 相同的字号，不再是之前的 4.5
MAIN_HEATMAP_YTICK_SHOW_INDEXES = None                   # None=按 EVERY 自动生成；想完全自定义填 list 如 [0,3,6,9,12,15,18,21,24,27,30,33]
MAIN_ENRICHMENT_HIST = dict(bins=35, density=True, edgecolor="white", linewidth=0.45, ci_alpha=0.07)
MAIN_OBSERVED_LINE = dict(color_key="red", linewidth=2.0)
MAIN_OCCUPANCY_OBSERVED = dict(linewidth=1.9, marker="o", markersize=2.3)
MAIN_OCCUPANCY_NULL_MEDIAN = dict(linestyle="--", linewidth=1.2)
MAIN_OCCUPANCY_NULL_CI = dict(alpha=0.60)
MAIN_PANEL_LABEL_POS = {
    "a": dict(x=-0.14, y=1.06),
    "b": dict(x=-0.16, y=1.06),
    "c": dict(x=-0.14, y=1.06),
    "d": dict(x=-0.16, y=1.06),
}
MAIN_PANEL_LABEL_STYLE = dict(fontsize=14, fontweight="bold", va="top", ha="left")
MAIN_STATS_TEXT = dict(fontsize=7.1, va="top", ha="left", position=(0.03, 0.96))
MAIN_OCCUPANCY_LEGEND_ORDER = [2, 1, 0]
MAIN_OCCUPANCY_LEGEND_LOC = "upper left"

# ==============================================================
# ★ 整张图 Figure 4 的图名（suptitle）：b/c 位置已互换、且 (b) 中注明和 Table S1 * 顺序一致
# ==============================================================

MAIN_FIGSIZE = (7.2, 6.5)
MAIN_FONT = {
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Microsoft YaHei", "SimHei", "Helvetica", "DejaVu Sans"],
    "font.size": 10,
    "axes.titlesize": 9.5,
    "axes.labelsize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,                         # ★ panel b 的 ytick 字号与此一致
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.9,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}
MAIN_GRID = dict(
    height_ratios=[1.1, 1.0],
    width_ratios=[1, 1],
    hspace=0.25,
    wspace=0.22,
)
MAIN_MARGINS = None                                           # 想固定留白填 dict(left=0.08, right=0.97, top=0.955, bottom=0.14)
MAIN_AXIS = dict(
    x_lim=(-24, 0),
    x_ticks=list(range(-24, 1, 6)),
    x_label="Time before debris-flow initiation (h)",
    final_hour_span=(-1, 0),
    final_hour_alpha=0.10,
    initiation_linewidth=0.75,
)
MAIN_GRID_LINES = dict(color=MAIN_COLORS["grid_line"], linewidth=0.45, alpha=MAIN_COLORS["grid_alpha"])
# ★ b/c 顺序已互换：b=heatmap；c=histogram；且 panel b 的 ylabel 中明确写了顺序与 Table S1 * 号候选簇顺序一致
MAIN_YLABELS = dict(
    panel_a="Cumulative count",
    panel_b=(
        "Event"
    ),
    panel_c="Probability density",
    panel_d="Mean state occupancy",
)
MAIN_OUT_FORMATS = ("svg", "pdf", "tiff", "png")
MAIN_OUT_DPI = 600
MAIN_OUT_STEM = "Figure_4"
# --------------------------------------------------------------------------------------------------------

def add_panel_label_local(ax, label, x=-0.14, y=1.08):
    ax.text(x, y, label, transform=ax.transAxes, **MAIN_PANEL_LABEL_STYLE)

def format_pvalue_local(value: float) -> str:
    return "<0.0001" if value < 0.0001 else f"={value:.4f}"


plt.rcParams.update(MAIN_FONT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fig = plt.figure(figsize=MAIN_FIGSIZE)
if MAIN_MARGINS is not None:
    fig.subplots_adjust(**MAIN_MARGINS)
grid = fig.add_gridspec(2, 2, **MAIN_GRID)

ax_a = fig.add_subplot(grid[0, 0])
ax_b = fig.add_subplot(grid[0, 1])
ax_c = fig.add_subplot(grid[1, 0])
ax_d = fig.add_subplot(grid[1, 1])

# ============== Panel a ==============
n_curves = len(cumulative)
line_cmap = plt.get_cmap(MAIN_COLORS["cumulative_colormap"], n_curves)
for index, curve in enumerate(cumulative):
    ax_a.plot(minute_axis, curve, color=line_cmap(index), **MAIN_CUMULATIVE)
ax_a.set_xlim(*MAIN_AXIS["x_lim"])
ax_a.set_xticks(MAIN_AXIS["x_ticks"])
ax_a.set_xlabel(MAIN_AXIS["x_label"])
ax_a.set_ylabel(MAIN_YLABELS["panel_a"])
# ax_a.grid(**MAIN_GRID_LINES)
add_panel_label_local(ax_a, "a", **MAIN_PANEL_LABEL_POS["a"])

# ============== Panel b — Hourly all-event heatmap（现在右上，b/c 已互换）==============
n_rows = len(heatmap_labels)
image = ax_b.imshow(
    hourly,
    aspect="auto",
    interpolation="nearest",
    cmap=MAIN_COLORS["heatmap_colormap"],
    extent=[-24, 0, n_rows - 0.5, -0.5],
    vmin=0,
)
ax_b.set_xlim(*MAIN_AXIS["x_lim"])
ax_b.set_xticks(MAIN_AXIS["x_ticks"])
ax_b.set_xlabel(MAIN_AXIS["x_label"])
ax_b.set_ylabel(MAIN_YLABELS["panel_b"])          # ★ ylabel 文字本身就带 "order follows * Table S1"

# ★ 关键点 1：ytick 位置仍然保留 0..n_rows-1 全部（让竖线仍对齐每行）
all_y_ticks = np.arange(n_rows)
ax_b.set_yticks(all_y_ticks)
# ★ 关键点 2：但 label 只在每 MAIN_HEATMAP_YTICK_EVERY 个位置显示真实名称，其它位置设为空字符串 —— 字号和其它 3 张图一样大
if MAIN_HEATMAP_YTICK_SHOW_INDEXES is None:
    show_indexes = list(range(0, n_rows, MAIN_HEATMAP_YTICK_EVERY))
else:
    show_indexes = list(MAIN_HEATMAP_YTICK_SHOW_INDEXES)
ytick_labels = [
    (heatmap_labels[i] if i in show_indexes else "")
    for i in range(n_rows)
]
ax_b.set_yticklabels(ytick_labels, fontsize=MAIN_HEATMAP_YTICK_FONTSIZE)

# ★ colorbar（精确位置垂直色条）
if MAIN_HEATMAP_COLORBAR.get("use_absolute_position", False):
    cbar_ax = fig.add_axes(MAIN_HEATMAP_COLORBAR["absolute_rect"])
    cbar = fig.colorbar(image, cax=cbar_ax, orientation=MAIN_HEATMAP_COLORBAR["orientation"])
    cbar.ax.tick_params(labelsize=MAIN_HEATMAP_COLORBAR["tick_labelsize"])
    cbar_ax.set_ylabel(
        MAIN_HEATMAP_COLORBAR["label"],
        fontsize=MAIN_HEATMAP_COLORBAR["ylabelsize"],
        fontfamily=MAIN_HEATMAP_COLORBAR["font_family"],
        labelpad=MAIN_HEATMAP_COLORBAR["labelpad"],
    )
else:
    RELATIVE_KEYS = ("orientation", "pad", "fraction", "aspect")
    cbar = fig.colorbar(
        image, ax=ax_b,
        **{k: MAIN_HEATMAP_COLORBAR[k] for k in RELATIVE_KEYS if k in MAIN_HEATMAP_COLORBAR},
    )
    cbar.ax.tick_params(labelsize=MAIN_HEATMAP_COLORBAR["tick_labelsize"])
    cbar.set_label(
        MAIN_HEATMAP_COLORBAR["label"],
        fontsize=MAIN_HEATMAP_COLORBAR["ylabelsize"],
        labelpad=MAIN_HEATMAP_COLORBAR["labelpad"],
    )
add_panel_label_local(ax_b, "b", **MAIN_PANEL_LABEL_POS["b"])

# ============== Panel c（现在左下，b/c 已互换）==============
enrichment = null_data["null_enrichment"]
enrichment = enrichment[np.isfinite(enrichment)]
ax_c.hist(
    enrichment,
    bins=MAIN_ENRICHMENT_HIST["bins"],
    density=MAIN_ENRICHMENT_HIST["density"],
    color=MAIN_COLORS["blue_light"],
    edgecolor=MAIN_ENRICHMENT_HIST["edgecolor"],
    linewidth=MAIN_ENRICHMENT_HIST["linewidth"],
)
ax_c.axvspan(
    null_data["null_enrichment_low"], null_data["null_enrichment_high"],
    color=MAIN_COLORS["blue"], alpha=MAIN_ENRICHMENT_HIST["ci_alpha"], linewidth=0,
)
ax_c.axvline(
    null_data["observed_enrichment"],
    color=MAIN_COLORS[MAIN_OBSERVED_LINE["color_key"]],
    linewidth=MAIN_OBSERVED_LINE["linewidth"],
)
ax_c.set_xlabel("Median final-hour enrichment")
ax_c.set_ylabel(MAIN_YLABELS["panel_c"])
ax_c.text(
    *MAIN_STATS_TEXT["position"],
    (
        f"Observed: {null_data['observed_enrichment']:.2f}x\n"
        f"Null median [95%]: {null_data['null_enrichment_median']:.2f} "
        f"[{null_data['null_enrichment_low']:.2f}, {null_data['null_enrichment_high']:.2f}]\n"
        f"Monte Carlo p {format_pvalue_local(null_data['p_enrichment'])}"
    ),
    transform=ax_c.transAxes,
    va=MAIN_STATS_TEXT["va"],
    ha=MAIN_STATS_TEXT["ha"],
    fontsize=MAIN_STATS_TEXT["fontsize"],
)
add_panel_label_local(ax_c, "c", **MAIN_PANEL_LABEL_POS["c"])

# ============== Panel d ==============
hours = null_data["time_hours"]
ax_d.fill_between(
    hours,
    null_data["null_occupancy_low"],
    null_data["null_occupancy_high"],
    color=MAIN_COLORS["grey"],
    alpha=MAIN_OCCUPANCY_NULL_CI["alpha"],
    linewidth=0,
    label="Null 95% interval",
)
ax_d.plot(
    hours, null_data["null_occupancy_median"],
    color=MAIN_COLORS["dark_grey"],
    linestyle=MAIN_OCCUPANCY_NULL_MEDIAN["linestyle"],
    linewidth=MAIN_OCCUPANCY_NULL_MEDIAN["linewidth"],
    label="Null median",
)
ax_d.plot(
    hours, null_data["observed_occupancy"],
    color=MAIN_COLORS["blue"],
    linewidth=MAIN_OCCUPANCY_OBSERVED["linewidth"],
    marker=MAIN_OCCUPANCY_OBSERVED["marker"],
    markersize=MAIN_OCCUPANCY_OBSERVED["markersize"],
    label="Observed alignment",
)
ax_d.axvspan(*MAIN_AXIS["final_hour_span"], color=MAIN_COLORS["orange"], alpha=MAIN_AXIS["final_hour_alpha"], linewidth=0)
ax_d.axvline(0, color=MAIN_COLORS["black"], linewidth=MAIN_AXIS["initiation_linewidth"])
ax_d.set_xlim(*MAIN_AXIS["x_lim"])
ax_d.set_xticks(MAIN_AXIS["x_ticks"])
ax_d.set_xlabel(MAIN_AXIS["x_label"])
ax_d.set_ylabel(MAIN_YLABELS["panel_d"])
handles, labels = ax_d.get_legend_handles_labels()
order = MAIN_OCCUPANCY_LEGEND_ORDER
ax_d.legend([handles[i] for i in order], [labels[i] for i in order], loc=MAIN_OCCUPANCY_LEGEND_LOC)
add_panel_label_local(ax_d, "d", **MAIN_PANEL_LABEL_POS["d"])



# ============== 导出 ==============
output_stem = OUTPUT_DIR / MAIN_OUT_STEM
for fmt in MAIN_OUT_FORMATS:
    ext = "." + fmt
    save_kwargs = dict(dpi=MAIN_OUT_DPI, bbox_inches="tight", facecolor="white")
    if fmt == "tiff":
        save_kwargs["pil_kwargs"] = {"compression": "tiff_lzw"}
    fig.savefig(str(output_stem) + ext, **save_kwargs)

try:
    display(fig)
except Exception:
    pass
from pathlib import Path

save_root = Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")
(save_root / "png").mkdir(parents=True, exist_ok=True)
(save_root / "tiff").mkdir(parents=True, exist_ok=True)

fig.savefig(save_root / "png" / "Fig 4.png", format="png", dpi=600, facecolor="white", transparent=False, bbox_inches="tight", pad_inches=0)
fig.savefig(save_root / "tiff" / "Fig 4.tiff", format="tiff", dpi=600, facecolor="white", transparent=False, bbox_inches="tight", pad_inches=0, pil_kwargs={"compression": "tiff_lzw"})
plt.close(fig)

print("Saved Figure 4 (b/c positions swapped, heatmap ytick every N events + full caption):")
for fmt in MAIN_OUT_FORMATS:
    print(f"  {str(output_stem)}.{fmt}")

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# ---------- 可编辑参数 ----------
MAIN_COLORS={"blue":"#0F4D92","blue_light":"#8CB9D9","orange":"#D97732","red":"#B64342","grey":"#CFCECE","dark_grey":"#4D4D4D","black":"#272727","cumulative_colormap":"tab20","heatmap_colormap":"YlOrRd","grid_line":"0.88","grid_alpha":0.8}
MAIN_CUMULATIVE=dict(linewidth=0.8,alpha=0.90)
MAIN_HEATMAP_COLORBAR=dict(use_absolute_position=True,absolute_rect=(0.92,0.525,0.015,0.355),orientation="vertical",label="Hourly count",tick_labelsize=8,ylabelsize=10,labelpad=6,font_family=["Arial"],pad=0.10,fraction=0.075,aspect=28)
MAIN_HEATMAP_YTICK_EVERY=2
MAIN_HEATMAP_YTICK_FONTSIZE=7.9
MAIN_HEATMAP_YTICK_SHOW_INDEXES=None
MAIN_ENRICHMENT_HIST=dict(bins=35,density=True,edgecolor="white",linewidth=0.45,ci_alpha=0.07)
MAIN_OBSERVED_LINE=dict(color_key="red",linewidth=2.0)
MAIN_OCCUPANCY_OBSERVED=dict(linewidth=1.9,marker="o",markersize=2.3)
MAIN_OCCUPANCY_NULL_MEDIAN=dict(linestyle="--",linewidth=1.2)
MAIN_OCCUPANCY_NULL_CI=dict(alpha=0.60)
MAIN_PANEL_LABEL_POS={"a":dict(x=-0.14,y=1.06),"b":dict(x=-0.16,y=1.06),"c":dict(x=-0.14,y=1.06),"d":dict(x=-0.16,y=1.06)}
MAIN_PANEL_LABEL_STYLE=dict(fontsize=14,fontweight="bold",va="top",ha="left")
MAIN_STATS_TEXT=dict(fontsize=7.1,va="top",ha="left",position=(0.03,0.96))
MAIN_OCCUPANCY_LEGEND_ORDER=[2,1,0]
MAIN_OCCUPANCY_LEGEND_LOC="upper left"
MAIN_FIGSIZE=(7.2,6.5)
MAIN_FONT={"font.family":"sans-serif","font.sans-serif":["Arial","Microsoft YaHei","SimHei","Helvetica","DejaVu Sans"],"font.size":10,"axes.titlesize":9.5,"axes.labelsize":10,"xtick.labelsize":8.5,"ytick.labelsize":8.5,"legend.fontsize":8,"axes.spines.top":True,"axes.spines.right":True,"axes.spines.bottom":True,"axes.spines.left":True,"axes.linewidth":0.9,"axes.edgecolor":"black","axes.facecolor":"white","figure.facecolor":"white","xtick.direction":"in","ytick.direction":"in","xtick.top":False,"ytick.right":False,"axes.grid":False,"legend.frameon":False,"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none"}
MAIN_GRID=dict(height_ratios=[1.1,1.0],width_ratios=[1,1],hspace=0.25,wspace=0.22)
MAIN_MARGINS=None
MAIN_AXIS=dict(x_lim=(-24,0),x_ticks=list(range(-24,1,6)),x_label="Time before debris-flow initiation (h)",final_hour_span=(-1,0),final_hour_alpha=0.10,initiation_linewidth=0.75)
MAIN_YLABELS=dict(panel_a="Cumulative count",panel_b="Event",panel_c="Probability density",panel_d="Mean state occupancy")
MAIN_OUT_FORMATS=("svg","pdf","tiff","png")
MAIN_OUT_DPI=600
MAIN_OUT_STEM="Figure_4"
SAVE_ROOT=Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")

def add_panel_label_local(ax,label,x=-0.14,y=1.08):
    ax.text(x,y,label,transform=ax.transAxes,**MAIN_PANEL_LABEL_STYLE)

def format_pvalue_local(value):
    return "<0.0001" if value<0.0001 else f"={value:.4f}"

def apply_closed_frame_local(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.9)
    ax.tick_params(axis="both",which="both",direction="out",bottom=True,left=True,top=False,right=False,labeltop=False,labelright=False,colors="black",pad=4)
    ax.tick_params(axis="both",which="major",length=3.5,width=0.8)
    ax.tick_params(axis="both",which="minor",length=2,width=0.6)
    ax.grid(False,which="both")

# ---------- 绘图 ----------
OUTPUT_DIR=Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
(SAVE_ROOT/"png").mkdir(parents=True,exist_ok=True)
(SAVE_ROOT/"tiff").mkdir(parents=True,exist_ok=True)

with plt.rc_context(MAIN_FONT):
    fig=plt.figure(figsize=MAIN_FIGSIZE)
    if MAIN_MARGINS is not None:
        fig.subplots_adjust(**MAIN_MARGINS)
    grid=fig.add_gridspec(2,2,**MAIN_GRID)
    ax_a=fig.add_subplot(grid[0,0])
    ax_b=fig.add_subplot(grid[0,1])
    ax_c=fig.add_subplot(grid[1,0])
    ax_d=fig.add_subplot(grid[1,1])

    # a：累计数量
    n_curves=len(cumulative)
    line_cmap=plt.get_cmap(MAIN_COLORS["cumulative_colormap"],n_curves)
    for index,curve in enumerate(cumulative):
        ax_a.plot(minute_axis,curve,color=line_cmap(index),**MAIN_CUMULATIVE)
    ax_a.set_xlim(*MAIN_AXIS["x_lim"])
    ax_a.set_xticks(MAIN_AXIS["x_ticks"])
    ax_a.set_xlabel(MAIN_AXIS["x_label"])
    ax_a.set_ylabel(MAIN_YLABELS["panel_a"])
    add_panel_label_local(ax_a,"a",**MAIN_PANEL_LABEL_POS["a"])

    # b：各事件逐小时热图
    n_rows=len(heatmap_labels)
    image=ax_b.imshow(hourly,aspect="auto",interpolation="nearest",cmap=MAIN_COLORS["heatmap_colormap"],extent=[-24,0,n_rows-0.5,-0.5],vmin=0)
    ax_b.set_xlim(*MAIN_AXIS["x_lim"])
    ax_b.set_xticks(MAIN_AXIS["x_ticks"])
    ax_b.set_xlabel(MAIN_AXIS["x_label"])
    ax_b.set_ylabel(MAIN_YLABELS["panel_b"])
    ax_b.set_yticks(np.arange(n_rows))
    show_indexes=set(range(0,n_rows,MAIN_HEATMAP_YTICK_EVERY)) if MAIN_HEATMAP_YTICK_SHOW_INDEXES is None else set(MAIN_HEATMAP_YTICK_SHOW_INDEXES)
    ax_b.set_yticklabels([heatmap_labels[i] if i in show_indexes else "" for i in range(n_rows)],fontsize=MAIN_HEATMAP_YTICK_FONTSIZE)
    if MAIN_HEATMAP_COLORBAR["use_absolute_position"]:
        cbar_ax=fig.add_axes(MAIN_HEATMAP_COLORBAR["absolute_rect"])
        cbar=fig.colorbar(image,cax=cbar_ax,orientation=MAIN_HEATMAP_COLORBAR["orientation"])
    else:
        relative_keys=("orientation","pad","fraction","aspect")
        cbar=fig.colorbar(image,ax=ax_b,**{k:MAIN_HEATMAP_COLORBAR[k] for k in relative_keys if k in MAIN_HEATMAP_COLORBAR})
    cbar.ax.tick_params(labelsize=MAIN_HEATMAP_COLORBAR["tick_labelsize"],direction="out",length=3,width=0.8)
    cbar.set_label(MAIN_HEATMAP_COLORBAR["label"],fontsize=MAIN_HEATMAP_COLORBAR["ylabelsize"],fontfamily=MAIN_HEATMAP_COLORBAR["font_family"],labelpad=MAIN_HEATMAP_COLORBAR["labelpad"])
    cbar.outline.set_visible(True)
    cbar.outline.set_edgecolor("black")
    cbar.outline.set_linewidth(0.8)
    add_panel_label_local(ax_b,"b",**MAIN_PANEL_LABEL_POS["b"])

    # c：末小时富集与零模型
    enrichment=np.asarray(null_data["null_enrichment"],dtype=float)
    enrichment=enrichment[np.isfinite(enrichment)]
    ax_c.hist(enrichment,bins=MAIN_ENRICHMENT_HIST["bins"],density=MAIN_ENRICHMENT_HIST["density"],color=MAIN_COLORS["blue_light"],edgecolor=MAIN_ENRICHMENT_HIST["edgecolor"],linewidth=MAIN_ENRICHMENT_HIST["linewidth"])
    ax_c.axvspan(null_data["null_enrichment_low"],null_data["null_enrichment_high"],color=MAIN_COLORS["blue"],alpha=MAIN_ENRICHMENT_HIST["ci_alpha"],linewidth=0)
    ax_c.axvline(null_data["observed_enrichment"],color=MAIN_COLORS[MAIN_OBSERVED_LINE["color_key"]],linewidth=MAIN_OBSERVED_LINE["linewidth"])
    ax_c.set_xlabel("Median final-hour enrichment")
    ax_c.set_ylabel(MAIN_YLABELS["panel_c"])
    stats_text=f"Observed: {null_data['observed_enrichment']:.2f}x\nNull median [95%]: {null_data['null_enrichment_median']:.2f} [{null_data['null_enrichment_low']:.2f}, {null_data['null_enrichment_high']:.2f}]\nMonte Carlo p {format_pvalue_local(null_data['p_enrichment'])}"
    ax_c.text(*MAIN_STATS_TEXT["position"],stats_text,transform=ax_c.transAxes,va=MAIN_STATS_TEXT["va"],ha=MAIN_STATS_TEXT["ha"],fontsize=MAIN_STATS_TEXT["fontsize"])
    add_panel_label_local(ax_c,"c",**MAIN_PANEL_LABEL_POS["c"])

    # d：平均状态占比
    hours=null_data["time_hours"]
    ax_d.fill_between(hours,null_data["null_occupancy_low"],null_data["null_occupancy_high"],color=MAIN_COLORS["grey"],alpha=MAIN_OCCUPANCY_NULL_CI["alpha"],linewidth=0,label="Null 95% interval")
    ax_d.plot(hours,null_data["null_occupancy_median"],color=MAIN_COLORS["dark_grey"],linestyle=MAIN_OCCUPANCY_NULL_MEDIAN["linestyle"],linewidth=MAIN_OCCUPANCY_NULL_MEDIAN["linewidth"],label="Null median")
    ax_d.plot(hours,null_data["observed_occupancy"],color=MAIN_COLORS["blue"],linewidth=MAIN_OCCUPANCY_OBSERVED["linewidth"],marker=MAIN_OCCUPANCY_OBSERVED["marker"],markersize=MAIN_OCCUPANCY_OBSERVED["markersize"],label="Observed alignment")
    ax_d.axvspan(*MAIN_AXIS["final_hour_span"],color=MAIN_COLORS["orange"],alpha=MAIN_AXIS["final_hour_alpha"],linewidth=0)
    ax_d.axvline(0,color=MAIN_COLORS["black"],linewidth=MAIN_AXIS["initiation_linewidth"])
    ax_d.set_xlim(*MAIN_AXIS["x_lim"])
    ax_d.set_xticks(MAIN_AXIS["x_ticks"])
    ax_d.set_xlabel(MAIN_AXIS["x_label"])
    ax_d.set_ylabel(MAIN_YLABELS["panel_d"])
    handles,labels=ax_d.get_legend_handles_labels()
    order=MAIN_OCCUPANCY_LEGEND_ORDER
    ax_d.legend([handles[i] for i in order],[labels[i] for i in order],loc=MAIN_OCCUPANCY_LEGEND_LOC,frameon=False)
    add_panel_label_local(ax_d,"d",**MAIN_PANEL_LABEL_POS["d"])

    # 四个子图统一全框，上、右无刻度
    for ax in (ax_a,ax_b,ax_c,ax_d):
        apply_closed_frame_local(ax)

    # 原目录导出
    output_stem=OUTPUT_DIR/MAIN_OUT_STEM
    for fmt in MAIN_OUT_FORMATS:
        save_kwargs=dict(dpi=MAIN_OUT_DPI,bbox_inches="tight",pad_inches=0,facecolor="white",transparent=False)
        if fmt=="tiff":
            save_kwargs["pil_kwargs"]={"compression":"tiff_lzw"}
        fig.savefig(str(output_stem)+"."+fmt,format=fmt,**save_kwargs)

    # 指定目录保存 Fig 4
    png_path=SAVE_ROOT/"png"/"Fig 4.png"
    tiff_path=SAVE_ROOT/"tiff"/"Fig 4.tiff"
    fig.savefig(png_path,format="png",dpi=MAIN_OUT_DPI,facecolor="white",transparent=False,bbox_inches="tight",pad_inches=0.1)
    fig.savefig(tiff_path,format="tiff",dpi=MAIN_OUT_DPI,facecolor="white",transparent=False,bbox_inches="tight",pad_inches=0.1,pil_kwargs={"compression":"tiff_lzw"})
    display(fig)
    plt.close(fig)

print("Saved Fig 4:")
print(png_path)
print(tiff_path)

## 7. Supplementary diagnostics

This supplementary figure provides three transparency checks that are not used
as independent statistical validation:

- **a**, cumulative curves for the highest soft-PCS cluster in each of the nine
  events for which no cluster passed all PCS hard criteria. These clusters are
  displayed as comparison states and are not classified as precursors.
- **b**, PCS distributions for the 35 finally selected event-clusters and all
  other 170 event-clusters. Because final membership is defined using the PCS
  procedure, this panel is descriptive; no between-group significance test is
  performed.
- **c**, cumulative event-time trajectories for all 170 event-clusters that were
  not retained by the final PCS procedure. Only the original cluster trajectories
  are shown, without a mean, median or uncertainty summary.

The figure is exported as PNG, PDF, SVG and 600-dpi TIFF together with source
data files.


In [ ]:
def load_pcs_scored_cluster_table() -> pd.DataFrame:
    """Recompute event-wise PCS for all 205 event-cluster rows."""
    metrics_path = DEC_RESULT_DIR / "precursor_curve_metrics" / "all_cluster_metrics_by_sheet.xlsx"
    require_file(metrics_path)
    metrics = pd.read_excel(metrics_path, sheet_name="all_metrics_wide")
    required = {
        "event", "cluster", "z_final", "rate_acceleration", "c_value",
        "terminal_growth_ratio", "terminal_growth_flag", "sample_count",
    }
    missing = required.difference(metrics.columns)
    if missing:
        raise KeyError(f"Missing columns in all-cluster table: {sorted(missing)}")

    metrics["event"] = metrics["event"].astype(str)
    metrics["cluster"] = pd.to_numeric(metrics["cluster"], errors="raise").astype(int)

    def stable_zscore(values: pd.Series) -> pd.Series:
        values = pd.to_numeric(values, errors="coerce")
        std = values.std(ddof=0)
        if not np.isfinite(std) or std <= 0:
            return pd.Series(np.zeros(len(values)), index=values.index, dtype=float)
        return (values - values.mean()) / std

    metrics["z_z_final"] = metrics.groupby("event")["z_final"].transform(stable_zscore)
    metrics["z_rate_acceleration"] = metrics.groupby("event")["rate_acceleration"].transform(stable_zscore)
    metrics["z_c_value"] = metrics.groupby("event")["c_value"].transform(stable_zscore)
    metrics["z_terminal_growth_ratio"] = metrics.groupby("event")["terminal_growth_ratio"].transform(stable_zscore)
    metrics["pcs"] = (
        metrics["z_z_final"]
        + metrics["z_rate_acceleration"]
        - metrics["z_c_value"]
        + metrics["z_terminal_growth_ratio"]
    )

    selected_pairs = set(zip(selected["event"], selected["cluster"]))
    metrics["pcs_group"] = [
        "PCS-selected state" if (event, cluster) in selected_pairs else "Other event-cluster"
        for event, cluster in zip(metrics["event"], metrics["cluster"])
    ]
    return metrics


def load_nonselected_soft_states(scored_metrics: pd.DataFrame) -> pd.DataFrame:
    """Load the best soft-PCS cluster for each event with no strict PCS selection."""
    pcs_table = pd.read_excel(PCS_WORKBOOK, sheet_name="pcs")
    pcs_table["event"] = pcs_table["event"].astype(str)
    failed = pcs_table.loc[pcs_table["selection_status"].ne("selected")].copy()
    failed["cluster"] = pd.to_numeric(failed["best_soft_cluster"], errors="raise").astype(int)
    failed["event_number"] = failed["event"].map(event_number)
    failed = failed.sort_values("event_number").reset_index(drop=True)
    if len(failed) != EXPECTED_EVENTS - EXPECTED_SELECTED:
        raise ValueError(f"Expected 9 non-selected events, found {len(failed)}.")

    lookup = scored_metrics.set_index(["event", "cluster"])["pcs"]
    failed["soft_pcs"] = [lookup.loc[(event, cluster)] for event, cluster in zip(failed["event"], failed["cluster"])]
    return failed


def build_nonselected_cumulative(failed: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    minute_axis = np.arange(1441, dtype=float) / 60.0 - 24.0
    curves = []
    labels_out = []
    for row in failed.itertuples(index=False):
        relative_hours, state_mask = load_event_state(row.event, int(row.cluster))
        valid = (relative_hours >= -24.0) & (relative_hours < 0.0)
        relative_hours = relative_hours[valid]
        state_mask = state_mask[valid]
        minute_index = np.floor((relative_hours + 24.0) * 60.0).astype(int)
        minute_index = np.clip(minute_index, 0, 1439)
        minute_counts = np.bincount(minute_index[state_mask], minlength=1440)
        curves.append(np.r_[0.0, np.cumsum(minute_counts, dtype=float)])
        labels_out.append(f"E{event_number(row.event)}–C{int(row.cluster)}")
    return np.stack(curves), minute_axis, labels_out


def build_all_other_cluster_cumulative(
    scored_metrics: pd.DataFrame,
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Build cumulative trajectories for all clusters not retained by PCS."""
    other = scored_metrics.loc[
        scored_metrics["pcs_group"].eq("Other event-cluster"),
        ["event", "cluster"],
    ].copy()
    other["event_number"] = other["event"].map(event_number)
    other = other.sort_values(["event_number", "cluster"]).reset_index(drop=True)

    minute_axis = np.arange(1441, dtype=float) / 60.0 - 24.0
    curves: List[np.ndarray] = []
    labels_out: List[str] = []
    for row in other.itertuples(index=False):
        relative_hours, state_mask = load_event_state(row.event, int(row.cluster))
        valid = (relative_hours >= -24.0) & (relative_hours < 0.0)
        relative_hours = relative_hours[valid]
        state_mask = state_mask[valid]
        minute_index = np.floor((relative_hours + 24.0) * 60.0).astype(int)
        minute_index = np.clip(minute_index, 0, 1439)
        minute_counts = np.bincount(minute_index[state_mask], minlength=1440)
        curves.append(np.r_[0.0, np.cumsum(minute_counts, dtype=float)])
        labels_out.append(f"E{event_number(row.event)}-C{int(row.cluster)}")

    if len(curves) != len(other):
        raise RuntimeError("Not all non-selected event-cluster curves were loaded.")
    return np.stack(curves), minute_axis, labels_out


In [ ]:
scored_metrics = load_pcs_scored_cluster_table()
failed_events = load_nonselected_soft_states(scored_metrics)
nonselected_curves, supplementary_minute_axis, nonselected_curve_labels = build_nonselected_cumulative(failed_events)
all_other_curves, all_other_minute_axis, all_other_curve_labels = build_all_other_cluster_cumulative(scored_metrics)
if not np.array_equal(supplementary_minute_axis, all_other_minute_axis):
    raise ValueError("Supplementary curve time axes do not match.")

print(f"Non-selected events: {len(failed_events)}")
print(f"PCS-selected event-clusters: {(scored_metrics['pcs_group'] == 'PCS-selected state').sum()}")
print(f"Other event-clusters: {(scored_metrics['pcs_group'] == 'Other event-cluster').sum()}")

# ----- CSV source data export (previously at end of plot_supplementary_diagnostics) -----
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
curve_data = pd.DataFrame({"time_before_initiation_h": supplementary_minute_axis})
for label, curve in zip(nonselected_curve_labels, nonselected_curves):
    curve_data[label] = curve
curve_data.to_csv(OUTPUT_DIR / "supp_panel_a_nonselected_cumulative_curves.csv", index=False)
scored_metrics[["event", "cluster", "pcs", "pcs_group"]].to_csv(
    OUTPUT_DIR / "supp_panel_b_all_cluster_pcs_values.csv", index=False,
)
all_other_data = pd.DataFrame(all_other_curves.T, columns=all_other_curve_labels)
all_other_data.insert(0, "time_before_initiation_h", supplementary_minute_axis)
all_other_data.to_csv(
    OUTPUT_DIR / "supp_panel_c_all_nonpcs_cluster_cumulative_curves.csv", index=False,
)


In [ ]:
# ---------- Editable: colours, panel-label positions, typography, dimensions, spacing ----------
SUPP_COLORS = {
    "blue":          "#0F4D92",
    "blue_light":    "#8CB9D9",
    "orange":        "#D97732",
    "red":           "#B64342",
    "grey":          "#CFCECE",
    "dark_grey":     "#4D4D4D",
    "black":         "#272727",
    "cumulative_a_colormap": "tab10",
    "cumulative_c_color":    "0.42",
    "boxplot_face":          ["#D97732", "#CFCECE"],
    "boxplot_edge":          "#4D4D4D",
    "boxplot_face_alpha":    0.75,
    "boxplot_median_line":   ("black", 1.2),
    "boxplot_whisker_line":  ("#4D4D4D", 0.9),
    "boxplot_zero_line":     ("0.75", 0.7),
    "scatter": dict(face_size=10, edgecolor="white", edgewidth=0.25, alpha=0.72, jitter=(-0.14, 0.14), seed=20260813),
    "grid":        "0.90",
    "grid_alpha":  0.8,
}
SUPP_PANEL_A = dict(linewidth=1.0)
SUPP_PANEL_A_LEGEND = dict(ncol=3, loc="upper left", fontsize=5.6, handlelength=1.5, columnspacing=0.8)
SUPP_PANEL_A_NLABEL = dict(position=(0.98, 0.04), ha="right", va="bottom")
SUPP_PANEL_B_BOX = dict(widths=0.46, showfliers=False, whis=1.5)
SUPP_PANEL_C = dict(linewidth=0.48, alpha=0.18, zorder=1)
SUPP_PANEL_LABEL_POS = {
    "a": dict(x=-0.16, y=1.08),
    "b": dict(x=-0.16, y=1.08),
    "c": dict(x=-0.16, y=1.08),
}
SUPP_PANEL_LABEL_STYLE = dict(fontsize=11, fontweight="bold", va="top", ha="left")
SUPP_FIGSIZE = (10.8, 3.15)
SUPP_FONT = {
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 8.5 + 1,
    "axes.titlesize": 9.5+ 1,
    "axes.labelsize": 8.5+ 1,
    "xtick.labelsize": 7.5+ 1,
    "ytick.labelsize": 7.2+ 1,
    "legend.fontsize": 7.0+ 1,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.9,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}
SUPP_WSPACE = 0.40
SUPP_AXIS = dict(
    x_lim=(-24, 0),
    x_ticks=list(range(-24, 1, 6)),
    x_label="Time before debris-flow initiation (h)",
    y_label_count="Cumulative count",
    y_label_pcs="Precursor-cluster score (PCS)",
)
SUPP_GRID_LINES = dict(color=SUPP_COLORS["grid"], linewidth=0.45, alpha=SUPP_COLORS["grid_alpha"])
SUPP_OUT_FORMATS = ("svg", "pdf", "tiff", "png")
SUPP_OUT_DPI = 600
SUPP_OUT_STEM = "Figure_S4"
# --------------------------------------------------------------------------------------------------------

def add_panel_label_local(ax, label, x=-0.16, y=1.08):
    ax.text(x, y, label, transform=ax.transAxes, **SUPP_PANEL_LABEL_STYLE)


plt.rcParams.update(SUPP_FONT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig, (ax_a, ax_b, ax_c) = plt.subplots(1, 3, figsize=SUPP_FIGSIZE, gridspec_kw={"wspace": SUPP_WSPACE})

# Panel a — Best soft-PCS cluster in each event with no strict PCS state
n_curves_a = len(nonselected_curves)
colors_a = plt.get_cmap(SUPP_COLORS["cumulative_a_colormap"], n_curves_a)
for index, (curve, label_text) in enumerate(zip(nonselected_curves, nonselected_curve_labels)):
    ax_a.plot(supplementary_minute_axis, curve, color=colors_a(index), linewidth=SUPP_PANEL_A["linewidth"], label=label_text)
ax_a.set_xlim(*SUPP_AXIS["x_lim"])
ax_a.set_xticks(SUPP_AXIS["x_ticks"])
ax_a.set_xlabel(SUPP_AXIS["x_label"])
ax_a.set_ylabel(SUPP_AXIS["y_label_count"])
ax_a.grid(**SUPP_GRID_LINES)
ax_a.legend(**SUPP_PANEL_A_LEGEND)
add_panel_label_local(ax_a, "a", **SUPP_PANEL_LABEL_POS["a"])

# Panel b — PCS distribution
groups = ["PCS-selected state", "Other event-cluster"]
values_b = [
    scored_metrics.loc[scored_metrics["pcs_group"].eq(group), "pcs"].dropna().to_numpy(float)
    for group in groups
]
positions = [1, 2]
box = ax_b.boxplot(
    values_b,
    positions=positions,
    widths=SUPP_PANEL_B_BOX["widths"],
    patch_artist=True,
    showfliers=SUPP_PANEL_B_BOX["showfliers"],
    whis=SUPP_PANEL_B_BOX["whis"],
    medianprops={"color": SUPP_COLORS["boxplot_median_line"][0], "linewidth": SUPP_COLORS["boxplot_median_line"][1]},
    whiskerprops={"color": SUPP_COLORS["boxplot_whisker_line"][0], "linewidth": SUPP_COLORS["boxplot_whisker_line"][1]},
    capprops={"color": SUPP_COLORS["boxplot_whisker_line"][0], "linewidth": SUPP_COLORS["boxplot_whisker_line"][1]},
)
for patch, color in zip(box["boxes"], SUPP_COLORS["boxplot_face"]):
    patch.set_facecolor(color)
    patch.set_edgecolor(SUPP_COLORS["boxplot_edge"])
    patch.set_alpha(SUPP_COLORS["boxplot_face_alpha"])

sc_cfg = SUPP_COLORS["scatter"]
rng = np.random.default_rng(sc_cfg["seed"])
for pos, group_values, color in zip(positions, values_b, SUPP_COLORS["boxplot_face"]):
    jitter = rng.uniform(sc_cfg["jitter"][0], sc_cfg["jitter"][1], size=len(group_values))
    ax_b.scatter(
        pos + jitter, group_values,
        s=sc_cfg["face_size"],
        facecolor=color,
        edgecolor=sc_cfg["edgecolor"],
        linewidth=sc_cfg["edgewidth"],
        alpha=sc_cfg["alpha"],
        zorder=3,
    )
labels_b = [f"PCS-selected\n(n={len(values_b[0])})", f"Other clusters\n(n={len(values_b[1])})"]
ax_b.set_xticks(positions)
ax_b.set_xticklabels(labels_b)
ax_b.set_ylabel(SUPP_AXIS["y_label_pcs"])
ax_b.axhline(0, color=SUPP_COLORS["boxplot_zero_line"][0], linewidth=SUPP_COLORS["boxplot_zero_line"][1], zorder=0)
ax_b.grid(axis="y", **SUPP_GRID_LINES)
add_panel_label_local(ax_b, "b", **SUPP_PANEL_LABEL_POS["b"])

# Panel c — Original trajectories for every non-PCS event-cluster
for curve in all_other_curves:
    ax_c.plot(
        supplementary_minute_axis, curve,
        color=SUPP_COLORS["cumulative_c_color"],
        linewidth=SUPP_PANEL_C["linewidth"],
        alpha=SUPP_PANEL_C["alpha"],
        zorder=SUPP_PANEL_C["zorder"],
    )
ax_c.set_xlim(*SUPP_AXIS["x_lim"])
ax_c.set_xticks(SUPP_AXIS["x_ticks"])
ax_c.set_xlabel(SUPP_AXIS["x_label"])
ax_c.set_ylabel(SUPP_AXIS["y_label_count"])
ax_c.text(
    *SUPP_PANEL_A_NLABEL["position"],
    f"n={len(all_other_curves)} event-clusters",
    transform=ax_c.transAxes,
    ha=SUPP_PANEL_A_NLABEL["ha"],
    va=SUPP_PANEL_A_NLABEL["va"],
    color=SUPP_COLORS["dark_grey"],
)
ax_c.grid(**SUPP_GRID_LINES)
add_panel_label_local(ax_c, "c", **SUPP_PANEL_LABEL_POS["c"])

output_stem = OUTPUT_DIR / SUPP_OUT_STEM
for fmt in SUPP_OUT_FORMATS:
    ext = "." + fmt
    save_kwargs = dict(dpi=SUPP_OUT_DPI, bbox_inches="tight", facecolor="white")
    if fmt == "tiff":
        save_kwargs["pil_kwargs"] = {"compression": "tiff_lzw"}
    fig.savefig(str(output_stem) + ext, **save_kwargs)

try:
    display(fig)
except Exception:
    pass
plt.close(fig)

print("Saved supplementary figure a-c:")
for fmt in SUPP_OUT_FORMATS:
    print(f"  {str(output_stem)}.{fmt}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. Editable settings
# ============================================================

# ---------- Figure ----------
FIGSIZE = (10.8, 3.15)
FIG_LEFT = 0.07
FIG_RIGHT = 0.985
FIG_BOTTOM = 0.20
FIG_TOP = 0.94
FIG_WSPACE = 0.25

# ---------- Font ----------
FONT_FAMILY = "Arial"
FONT_BASE = 9.5+2
FONT_AXIS_LABEL = 9.5+2
FONT_TICK = 8.5+2
FONT_LEGEND = 8.0+1
FONT_NLABEL = 7.5+2
FONT_PANEL_LABEL = 11.0+4

# ---------- Panel-label positions ----------
PANEL_LABEL_POS = {
    "a": (-0.15, 1.08),
    "b": (-0.15, 1.08),
    "c": (-0.15, 1.08),
}
PANEL_LABEL_WEIGHT = "bold"

# ---------- Colors ----------
COLOR_ORANGE = "#D97732"
COLOR_GREY = "#CFCECE"
COLOR_DARK_GREY = "#4D4D4D"
COLOR_BLACK = "#272727"
COLOR_GRID = "0.90"
COLOR_PANEL_C = "0.42"
PANEL_A_CMAP = "tab10"

# ---------- Grid ----------
GRID_LINEWIDTH = 0.45
GRID_ALPHA = 0.80

# ---------- Panel a ----------
A_LINEWIDTH = 1.0
A_LEGEND_NCOL = 3
A_LEGEND_LOC = "upper left"
A_LEGEND_HANDLELENGTH = 1.5
A_LEGEND_COLUMNSPACING = 0.8
A_NLABEL_POS = (0.98, 0.04)

# ---------- Panel b ----------
B_BOX_WIDTH = 0.46
B_SHOWFLIERS = False
B_WHIS = 1.5
B_BOX_ALPHA = 0.75
B_MEDIAN_LINEWIDTH = 1.2
B_WHISKER_LINEWIDTH = 0.9
B_ZERO_LINEWIDTH = 0.7
B_SCATTER_SIZE = 10
B_SCATTER_ALPHA = 0.72
B_SCATTER_EDGEWIDTH = 0.25
B_SCATTER_JITTER = (-0.14, 0.14)
B_SCATTER_SEED = 20260813
B_NLABEL_Y = 0.96   # n=35 / n=170 在图内的纵向位置

# ---------- Panel c ----------
C_LINEWIDTH = 0.48
C_ALPHA = 0.18
C_NLABEL_POS = (0.98, 0.04)

# ---------- Axes ----------
X_LIM = (-24, 0)
X_TICKS = list(range(-24, 1, 6))
X_LABEL = "Time before debris-flow initiation (h)"
Y_LABEL_COUNT = "Cumulative count"
Y_LABEL_PCS = "Precursor-cluster score (PCS)"
B_XLABEL = "Event cluster type"
# ---------- Export ----------
OUT_STEM = "Figure_S4"
OUT_DPI = 600

# ============================================================
# 2. Matplotlib style
# ============================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [FONT_FAMILY, "Microsoft YaHei", "SimHei", "Helvetica", "DejaVu Sans"],
    "font.size": FONT_BASE,
    "axes.labelsize": FONT_AXIS_LABEL,
    "xtick.labelsize": FONT_TICK,
    "ytick.labelsize": FONT_TICK,
    "legend.fontsize": FONT_LEGEND,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.9,
    "legend.frameon": False,
})

# ============================================================
# 3. Helper functions
# ============================================================

def add_panel_label(ax, label):
    x, y = PANEL_LABEL_POS[label]
    ax.text(x, y, label, transform=ax.transAxes, fontsize=FONT_PANEL_LABEL,
            fontweight=PANEL_LABEL_WEIGHT, ha="left", va="top")

def set_common_axis(ax, xlabel="", ylabel=""):
    ax.set_xlabel(xlabel, fontsize=FONT_AXIS_LABEL)
    ax.set_ylabel(ylabel, fontsize=FONT_AXIS_LABEL)
    ax.tick_params(axis="both", labelsize=FONT_TICK)

# ============================================================
# 4. Create figure
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fig, (ax_a, ax_b, ax_c) = plt.subplots(1, 3, figsize=FIGSIZE)
fig.subplots_adjust(left=FIG_LEFT, right=FIG_RIGHT, bottom=FIG_BOTTOM,
                    top=FIG_TOP, wspace=FIG_WSPACE)

# ============================================================
# 5. Panel a
# Highest-scoring states in events without strict PCS selection
# ============================================================

n_curves_a = len(nonselected_curves)
colors_a = plt.get_cmap(PANEL_A_CMAP, n_curves_a)

for i, (curve, label_text) in enumerate(zip(nonselected_curves, nonselected_curve_labels)):
    ax_a.plot(supplementary_minute_axis, curve, color=colors_a(i),
              linewidth=A_LINEWIDTH, label=label_text)

ax_a.set_xlim(*X_LIM)
ax_a.set_xticks(X_TICKS)
set_common_axis(ax_a, X_LABEL, Y_LABEL_COUNT)

# ax_a.grid(color=COLOR_GRID, linewidth=GRID_LINEWIDTH, alpha=GRID_ALPHA)

ax_a.legend(ncol=A_LEGEND_NCOL, loc=A_LEGEND_LOC,
            handlelength=A_LEGEND_HANDLELENGTH,
            columnspacing=A_LEGEND_COLUMNSPACING,
            fontsize=FONT_LEGEND)

add_panel_label(ax_a, "a")

# ============================================================
# 6. Panel b
# PCS distribution
# ============================================================

groups = ["PCS-selected state", "Other event-cluster"]

values_b = [
    scored_metrics.loc[scored_metrics["pcs_group"].eq(group), "pcs"].dropna().to_numpy(float)
    for group in groups
]

positions = [1, 2]

box = ax_b.boxplot(
    values_b, positions=positions, widths=B_BOX_WIDTH, patch_artist=True,
    showfliers=B_SHOWFLIERS, whis=B_WHIS,
    medianprops={"color": COLOR_BLACK, "linewidth": B_MEDIAN_LINEWIDTH},
    whiskerprops={"color": COLOR_DARK_GREY, "linewidth": B_WHISKER_LINEWIDTH},
    capprops={"color": COLOR_DARK_GREY, "linewidth": B_WHISKER_LINEWIDTH}
)

box_colors = [COLOR_ORANGE, COLOR_GREY]

for patch, color in zip(box["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_edgecolor(COLOR_DARK_GREY)
    patch.set_alpha(B_BOX_ALPHA)

rng = np.random.default_rng(B_SCATTER_SEED)

for pos, values, color in zip(positions, values_b, box_colors):
    jitter = rng.uniform(B_SCATTER_JITTER[0], B_SCATTER_JITTER[1], size=len(values))
    ax_b.scatter(pos + jitter, values, s=B_SCATTER_SIZE,
                 facecolor=color, edgecolor="white",
                 linewidth=B_SCATTER_EDGEWIDTH,
                 alpha=B_SCATTER_ALPHA, zorder=3)

# x-axis labels only show group names
ax_b.set_xticks(positions)
ax_b.set_xticklabels(["PCS-selected (35)", "Other clusters (170)"], fontsize=FONT_TICK)

set_common_axis(ax_b, B_XLABEL, Y_LABEL_PCS)

# Sample-size labels inside the panel
# for pos, values in zip(positions, values_b):
#     ax_b.text(pos, B_NLABEL_Y, f"n={len(values)}",
#               transform=ax_b.get_xaxis_transform(),
#               ha="center", va="top",
#               fontsize=FONT_NLABEL,
#               color=COLOR_DARK_GREY)

ax_b.axhline(0, color="0.75", linewidth=B_ZERO_LINEWIDTH, zorder=0)
# ax_b.grid(axis="y", color=COLOR_GRID, linewidth=GRID_LINEWIDTH, alpha=GRID_ALPHA)

add_panel_label(ax_b, "b")

# ============================================================
# 7. Panel c
# All non-selected event-cluster cumulative trajectories
# ============================================================

for curve in all_other_curves:
    ax_c.plot(supplementary_minute_axis, curve,
              color=COLOR_PANEL_C,
              linewidth=C_LINEWIDTH,
              alpha=C_ALPHA,
              zorder=1)

ax_c.set_xlim(*X_LIM)
ax_c.set_xticks(X_TICKS)
set_common_axis(ax_c, X_LABEL, Y_LABEL_COUNT)

ax_c.text(*C_NLABEL_POS, f"n={len(all_other_curves)} event-clusters",
          transform=ax_c.transAxes, ha="right", va="bottom",
          color=COLOR_DARK_GREY, fontsize=FONT_NLABEL)
# 
# ax_c.grid(color=COLOR_GRID, linewidth=GRID_LINEWIDTH, alpha=GRID_ALPHA)

add_panel_label(ax_c, "c")

# ============================================================
# 8. Export
# ============================================================

output_stem = OUTPUT_DIR / OUT_STEM

fig.savefig(f"{output_stem}.png",
            dpi=OUT_DPI,
            bbox_inches="tight",
            facecolor="white")

fig.savefig(f"{output_stem}.tiff",
            dpi=OUT_DPI,
            bbox_inches="tight",
            facecolor="white",
            pil_kwargs={"compression": "tiff_lzw"})

try:
    display(fig)
except Exception:
    plt.show()

plt.close(fig)

print("Saved:")
print(f"  {output_stem}.png")
print(f"  {output_stem}.tiff")

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# ---------- 参数 ----------
FIGSIZE=(10.8,3.15)
FIG_LEFT=0.07
FIG_RIGHT=0.985
FIG_BOTTOM=0.20
FIG_TOP=0.94
FIG_WSPACE=0.25
FONT_FAMILY="Arial"
FONT_BASE=11.5
FONT_AXIS_LABEL=11.5
FONT_TICK=10.5
FONT_LEGEND=9.0
FONT_NLABEL=9.5
FONT_PANEL_LABEL=15.0
PANEL_LABEL_POS={"a":(-0.15,1.08),"b":(-0.15,1.08),"c":(-0.15,1.08)}
PANEL_LABEL_WEIGHT="bold"
COLOR_ORANGE="#D97732"
COLOR_GREY="#CFCECE"
COLOR_DARK_GREY="#4D4D4D"
COLOR_BLACK="#272727"
COLOR_PANEL_C="0.42"
PANEL_A_CMAP="tab10"
A_LINEWIDTH=1.0
A_LEGEND_NCOL=3
A_LEGEND_LOC="upper left"
A_LEGEND_HANDLELENGTH=1.5
A_LEGEND_COLUMNSPACING=0.8
B_BOX_WIDTH=0.46
B_SHOWFLIERS=False
B_WHIS=1.5
B_BOX_ALPHA=0.75
B_MEDIAN_LINEWIDTH=1.2
B_WHISKER_LINEWIDTH=0.9
B_ZERO_LINEWIDTH=0.7
B_SCATTER_SIZE=10
B_SCATTER_ALPHA=0.72
B_SCATTER_EDGEWIDTH=0.25
B_SCATTER_JITTER=(-0.14,0.14)
B_SCATTER_SEED=20260813
C_LINEWIDTH=0.48
C_ALPHA=0.18
C_NLABEL_POS=(0.98,0.04)
X_LIM=(-24,0)
X_TICKS=list(range(-24,1,6))
X_LABEL="Time before debris-flow initiation (h)"
Y_LABEL_COUNT="Cumulative count"
Y_LABEL_PCS="Precursor-cluster score (PCS)"
B_XLABEL="Event cluster type"
OUT_DPI=600
SAVE_ROOT=Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")
PNG_PATH=SAVE_ROOT/"png"/"Fig S4.png"
TIFF_PATH=SAVE_ROOT/"tiff"/"Fig S4.tiff"
STYLE={"font.family":"sans-serif","font.sans-serif":[FONT_FAMILY,"Microsoft YaHei","SimHei","Helvetica","DejaVu Sans"],"font.size":FONT_BASE,"axes.labelsize":FONT_AXIS_LABEL,"xtick.labelsize":FONT_TICK,"ytick.labelsize":FONT_TICK,"legend.fontsize":FONT_LEGEND,"axes.spines.top":True,"axes.spines.right":True,"axes.spines.bottom":True,"axes.spines.left":True,"axes.linewidth":0.9,"axes.edgecolor":"black","axes.facecolor":"white","figure.facecolor":"white","axes.grid":False,"xtick.direction":"out","ytick.direction":"out","xtick.top":False,"ytick.right":False,"legend.frameon":False}

def add_panel_label(ax,label):
    x,y=PANEL_LABEL_POS[label]
    ax.text(x,y,label,transform=ax.transAxes,fontsize=FONT_PANEL_LABEL,fontweight=PANEL_LABEL_WEIGHT,ha="left",va="top")

def set_common_axis(ax,xlabel="",ylabel=""):
    ax.set_xlabel(xlabel,fontsize=FONT_AXIS_LABEL)
    ax.set_ylabel(ylabel,fontsize=FONT_AXIS_LABEL)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.9)
    ax.tick_params(axis="both",which="both",direction="out",labelsize=FONT_TICK,bottom=True,left=True,top=False,right=False,labeltop=False,labelright=False,length=3.5,width=0.8)
    ax.grid(False,which="both")

# ---------- 绘图 ----------
PNG_PATH.parent.mkdir(parents=True,exist_ok=True)
TIFF_PATH.parent.mkdir(parents=True,exist_ok=True)

with plt.rc_context(STYLE):
    fig,(ax_a,ax_b,ax_c)=plt.subplots(1,3,figsize=FIGSIZE)
    fig.subplots_adjust(left=FIG_LEFT,right=FIG_RIGHT,bottom=FIG_BOTTOM,top=FIG_TOP,wspace=FIG_WSPACE)

    # a：未通过严格PCS筛选事件中的最高分状态
    n_curves_a=len(nonselected_curves)
    colors_a=plt.get_cmap(PANEL_A_CMAP,n_curves_a)
    for i,(curve,label_text) in enumerate(zip(nonselected_curves,nonselected_curve_labels)):
        ax_a.plot(supplementary_minute_axis,curve,color=colors_a(i),linewidth=A_LINEWIDTH,label=label_text)
    ax_a.set_xlim(*X_LIM)
    ax_a.set_xticks(X_TICKS)
    set_common_axis(ax_a,X_LABEL,Y_LABEL_COUNT)
    ax_a.legend(ncol=A_LEGEND_NCOL,loc=A_LEGEND_LOC,handlelength=A_LEGEND_HANDLELENGTH,columnspacing=A_LEGEND_COLUMNSPACING,fontsize=FONT_LEGEND,frameon=False)
    add_panel_label(ax_a,"a")

    # b：PCS分布
    groups=["PCS-selected state","Other event-cluster"]
    values_b=[scored_metrics.loc[scored_metrics["pcs_group"].eq(group),"pcs"].dropna().to_numpy(float) for group in groups]
    positions=[1,2]
    box=ax_b.boxplot(values_b,positions=positions,widths=B_BOX_WIDTH,patch_artist=True,showfliers=B_SHOWFLIERS,whis=B_WHIS,medianprops={"color":COLOR_BLACK,"linewidth":B_MEDIAN_LINEWIDTH},whiskerprops={"color":COLOR_DARK_GREY,"linewidth":B_WHISKER_LINEWIDTH},capprops={"color":COLOR_DARK_GREY,"linewidth":B_WHISKER_LINEWIDTH})
    box_colors=[COLOR_ORANGE,COLOR_GREY]
    for patch,color in zip(box["boxes"],box_colors):
        patch.set_facecolor(color)
        patch.set_edgecolor(COLOR_DARK_GREY)
        patch.set_alpha(B_BOX_ALPHA)
    rng=np.random.default_rng(B_SCATTER_SEED)
    for pos,values,color in zip(positions,values_b,box_colors):
        jitter=rng.uniform(*B_SCATTER_JITTER,size=len(values))
        ax_b.scatter(pos+jitter,values,s=B_SCATTER_SIZE,facecolor=color,edgecolor="white",linewidth=B_SCATTER_EDGEWIDTH,alpha=B_SCATTER_ALPHA,zorder=3)
    ax_b.set_xticks(positions)
    ax_b.set_xticklabels([f"PCS-selected ({len(values_b[0])})",f"Other clusters ({len(values_b[1])})"],fontsize=FONT_TICK)
    set_common_axis(ax_b,B_XLABEL,Y_LABEL_PCS)
    ax_b.axhline(0,color="0.75",linewidth=B_ZERO_LINEWIDTH,zorder=0)
    add_panel_label(ax_b,"b")

    # c：其他事件状态的累计轨迹
    for curve in all_other_curves:
        ax_c.plot(supplementary_minute_axis,curve,color=COLOR_PANEL_C,linewidth=C_LINEWIDTH,alpha=C_ALPHA,zorder=1)
    ax_c.set_xlim(*X_LIM)
    ax_c.set_xticks(X_TICKS)
    set_common_axis(ax_c,X_LABEL,Y_LABEL_COUNT)
    ax_c.text(*C_NLABEL_POS,f"n={len(all_other_curves)} event-clusters",transform=ax_c.transAxes,ha="right",va="bottom",color=COLOR_DARK_GREY,fontsize=FONT_NLABEL)
    add_panel_label(ax_c,"c")

    # 保存：600 dpi，无额外padding
    fig.savefig(PNG_PATH,format="png",dpi=600,bbox_inches="tight",pad_inches=0,facecolor="white",transparent=False)
    fig.savefig(TIFF_PATH,format="tiff",dpi=600,bbox_inches="tight",pad_inches=0,facecolor="white",transparent=False,pil_kwargs={"compression":"tiff_lzw"})
    display(fig)
    plt.close(fig)

print("Saved Fig S4:")
print(PNG_PATH)
print(TIFF_PATH)